In [1]:
# Main purposes of this temporary notebook:
# 1. Check the integrity of the electricity production data downloaded from entso-e's transparency platform
# 2. Create and examine temporary DataFrames for debugging Python code

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from timeit import default_timer as timer

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display all columns
pd.set_option('display.max_columns', None)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.figsize'] = (11, 7)
plt.style.use('fivethirtyeight')


# 1. Data integrity and quality check for concatenated data

In [15]:
# Check raw CSV files downloaded from the transparency platform

input_folder = '/Users/shuxu_ds/workspace/neuefische_DS_bootcamp/capstone_solar/capstone_solar_energy/data/transparency/raw/'
csv_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.csv')])
print(csv_files)

for csv_file in csv_files:
    csv_file = os.path.join(input_folder, csv_file)
    df = pd.read_csv(csv_file, low_memory=False)
    # df.head()
    print(df.columns[19])
    print(df.shape)
    df.info()

['2015.csv', '2016.csv', '2017.csv', '2018.csv', '2019.csv', '2020.csv', '2021.csv', '2022.csv', '2023.csv']
Solar  - Actual Aggregated [MW]
(35044, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35044 entries, 0 to 35043
Data columns (total 23 columns):
 #   Column                                                     Non-Null Count  Dtype  
---  ------                                                     --------------  -----  
 0   Area                                                       35044 non-null  object 
 1   MTU                                                        35044 non-null  object 
 2   Biomass  - Actual Aggregated [MW]                          35040 non-null  float64
 3   Fossil Brown coal/Lignite  - Actual Aggregated [MW]        35040 non-null  float64
 4   Fossil Coal-derived gas  - Actual Aggregated [MW]          35040 non-null  float64
 5   Fossil Gas  - Actual Aggregated [MW]                       35040 non-null  float64
 6   Fossil Hard coal  - Actual Ag

In [3]:
df_long = pd.read_csv('../data/transparency/result_20150101-20230716.csv', low_memory=False)
df_long.head(1)
df_long.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315588 entries, 0 to 315587
Data columns (total 23 columns):
 #   Column                                                     Non-Null Count   Dtype 
---  ------                                                     --------------   ----- 
 0   Area                                                       315588 non-null  object
 1   MTU                                                        315588 non-null  object
 2   Biomass  - Actual Aggregated [MW]                          315551 non-null  object
 3   Fossil Brown coal/Lignite  - Actual Aggregated [MW]        315551 non-null  object
 4   Fossil Coal-derived gas  - Actual Aggregated [MW]          315552 non-null  object
 5   Fossil Gas  - Actual Aggregated [MW]                       315551 non-null  object
 6   Fossil Hard coal  - Actual Aggregated [MW]                 315551 non-null  object
 7   Fossil Oil  - Actual Aggregated [MW]                       315552 non-null  object
 8   Foss

In [4]:
df_long.shape

(315588, 23)

In [5]:
print(df_long['Area'].nunique())

1


In [6]:
df_long['Solar  - Actual Aggregated [MW]'].describe()

count     315552
unique     37520
top          0.0
freq      127161
Name: Solar  - Actual Aggregated [MW], dtype: object

In [7]:
print(df_long['Solar  - Actual Aggregated [MW]'].isna().sum())
df_long['Solar  - Actual Aggregated [MW]'].describe()

36


count     315552
unique     37520
top          0.0
freq      127161
Name: Solar  - Actual Aggregated [MW], dtype: object

In [8]:
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315588 entries, 0 to 315587
Data columns (total 23 columns):
 #   Column                                                     Non-Null Count   Dtype 
---  ------                                                     --------------   ----- 
 0   Area                                                       315588 non-null  object
 1   MTU                                                        315588 non-null  object
 2   Biomass  - Actual Aggregated [MW]                          315551 non-null  object
 3   Fossil Brown coal/Lignite  - Actual Aggregated [MW]        315551 non-null  object
 4   Fossil Coal-derived gas  - Actual Aggregated [MW]          315552 non-null  object
 5   Fossil Gas  - Actual Aggregated [MW]                       315551 non-null  object
 6   Fossil Hard coal  - Actual Aggregated [MW]                 315551 non-null  object
 7   Fossil Oil  - Actual Aggregated [MW]                       315552 non-null  object
 8   Foss

# 2. Data integrity and quality check for condensed hourly data

In [9]:
df_condense = pd.read_csv('../data/transparency/transparency_20150101-20230716_hourly.csv', low_memory=False)
df_condense.head(1)
df_condense.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74879 entries, 0 to 74878
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   area                                  74879 non-null  object 
 1   mtu                                   74879 non-null  object 
 2   biomass_mwh                           74879 non-null  float64
 3   fossil_brown_coal_mwh                 74879 non-null  float64
 4   fossil_coal_derived_gas_mwh           74879 non-null  float64
 5   fossil_gas_mwh                        74879 non-null  float64
 6   fossil_hard_coal_mwh                  74879 non-null  float64
 7   fossil_oil_mwh                        74879 non-null  float64
 8   fossil_oil_shale_mwh                  74879 non-null  float64
 9   fossil_peat_mwh                       74879 non-null  float64
 10  geothermal_mwh                        74879 non-null  float64
 11  hydro_pumped_st

In [10]:
# Check the data types and their distributions in the 'Solar_MW' column
data_types_counts = df_condense['solar_mwh'].apply(type).value_counts()

print("Data Types in 'solar_mwh' column and their counts:")
print(data_types_counts)

# If the column has mixed data types, you can see the exact data distributions using:
data_distributions = df_condense['solar_mwh'].value_counts()

print("\nData Distributions in 'solar_mwh' column:")
print(data_distributions)

Data Types in 'solar_mwh' column and their counts:
solar_mwh
<class 'float'>    74879
Name: count, dtype: int64

Data Distributions in 'solar_mwh' column:
solar_mwh
0.00       31598
0.25         431
0.50         210
0.75         190
1.00         154
           ...  
9667.00        1
9492.00        1
7513.50        1
4702.75        1
1511.25        1
Name: count, Length: 30693, dtype: int64


In [11]:
df = df_condense.copy()

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74879 entries, 0 to 74878
Data columns (total 26 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   area                                  74879 non-null  object 
 1   mtu                                   74879 non-null  object 
 2   biomass_mwh                           74879 non-null  float64
 3   fossil_brown_coal_mwh                 74879 non-null  float64
 4   fossil_coal_derived_gas_mwh           74879 non-null  float64
 5   fossil_gas_mwh                        74879 non-null  float64
 6   fossil_hard_coal_mwh                  74879 non-null  float64
 7   fossil_oil_mwh                        74879 non-null  float64
 8   fossil_oil_shale_mwh                  74879 non-null  float64
 9   fossil_peat_mwh                       74879 non-null  float64
 10  geothermal_mwh                        74879 non-null  float64
 11  hydro_pumped_st

In [13]:
print(df.shape)
print(df.columns)
df.describe().T

(74879, 26)
Index(['area', 'mtu', 'biomass_mwh', 'fossil_brown_coal_mwh',
       'fossil_coal_derived_gas_mwh', 'fossil_gas_mwh', 'fossil_hard_coal_mwh',
       'fossil_oil_mwh', 'fossil_oil_shale_mwh', 'fossil_peat_mwh',
       'geothermal_mwh', 'hydro_pumped_storage_aggregated_mwh',
       'hydro_pumped_storage_consumption_mwh',
       'hydro_run_of_river_and_poundage_mwh', 'hydro_water_reservoir_mwh',
       'marine_mwh', 'nuclear_mwh', 'other_mwh', 'other_renewable_mwh',
       'solar_mwh', 'waste_mwh', 'wind_offshore_mwh', 'wind_onshore_mwh',
       'date', 'hour', 'total_energy_generation_mwh'],
      dtype='object')


,count,mean,std,min,25%,50%,75%,max
biomass_mwh,74879.0,4.493048e+03,312.412053,0.0,4.329500e+03,4531.50,4.731250e+03,5167.00
fossil_brown_coal_mwh,74879.0,1.272760e+04,3606.732051,0.0,1.076462e+04,13505.75,1.543925e+04,19777.75
fossil_coal_derived_gas_mwh,74879.0,1.906699e+02,216.945809,0.0,0.000000e+00,0.00,4.050000e+02,753.50
fossil_gas_mwh,74879.0,4.510453e+03,2802.762193,0.0,2.254000e+03,3833.50,6.318500e+03,15033.25
fossil_hard_coal_mwh,74879.0,6.945407e+03,4310.568503,0.0,3.091250e+03,6182.25,1.027162e+04,19143.25
fossil_oil_mwh,74879.0,3.113211e+02,123.682768,0.0,2.000000e+02,293.00,4.015000e+02,1136.00
fossil_oil_shale_mwh,74879.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.00,0.000000e+00,0.00
fossil_peat_mwh,74879.0,0.000000e+00,0.000000,0.0,0.000000e+00,0.00,0.000000e+00,0.00
geothermal_mwh,74879.0,2.016376e+01,5.447995,0.0,1.650000e+01,20.00,2.400000e+01,34.00
hydro_pumped_storage_aggregated_mwh,74879.0,1.090392e+03,1203.844668,0.0,2.032500e+02,623.25,1.617250e+03,9163.00


In [14]:
df.head(1)

,area,mtu,biomass_mwh,fossil_brown_coal_mwh,fossil_coal_derived_gas_mwh,fossil_gas_mwh,fossil_hard_coal_mwh,fossil_oil_mwh,fossil_oil_shale_mwh,fossil_peat_mwh,geothermal_mwh,hydro_pumped_storage_aggregated_mwh,hydro_pumped_storage_consumption_mwh,hydro_run_of_river_and_poundage_mwh,hydro_water_reservoir_mwh,marine_mwh,nuclear_mwh,other_mwh,other_renewable_mwh,solar_mwh,waste_mwh,wind_offshore_mwh,wind_onshore_mwh,date,hour,total_energy_generation_mwh
0,Germany (DE),20150101 00:00-01:00,3997.5,15687.25,0.0,1226.25,3219.75,176.5,0.0,0.0,10.0,913.75,588.0,1148.0,10.25,0.0,10710.5,4567.5,123.0,0.0,298.25,521.0,8849.75,20150101,0,50545.5
